Enable GPU T4 + Internet. Fill API_KEY, then run. No Dataset upload required.

In [ ]:
import os, sys, subprocess, tarfile
from pathlib import Path

API_KEY = "YOUR_NVIDIA_API_KEY_HERE"
REPO = "https://github.com/HuyL13/test_debate_agents.git"
BRANCH = "feat/courtroom-kaggle-bge"
RESUME = None  # Optional: path to last.pt created by this exact source/config.
PROJECT = Path("/kaggle/working/courtroom-github")
assert API_KEY.strip() and API_KEY != "YOUR_NVIDIA_API_KEY_HERE", "Fill in your NVIDIA API key"

def run(*args):
    subprocess.run(list(args), check=True)

if not PROJECT.exists():
    run("git", "clone", "--single-branch", "--branch", BRANCH, REPO, str(PROJECT))
os.chdir(PROJECT)
print("Source commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
run(sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt")
os.environ["NVIDIA_API_KEY"] = API_KEY.strip()
os.environ["NVIDIA_BASE_URL"] = "https://integrate.api.nvidia.com/v1"
os.environ["NVIDIA_MODEL"] = "meta/llama-3.2-11b-vision-instruct"
os.environ["PYTHONUNBUFFERED"] = "1"
import torch
assert torch.cuda.is_available(), "Enable GPU T4 in Kaggle Settings"
run(sys.executable, "-u", "scripts/prepare_courtroom_kaggle.py", "--config", "configs/courtroom_rl.v4.kaggle.yaml")
run(sys.executable, "-u", "scripts/check_kaggle_gpu.py", "--config", "configs/courtroom_rl.v4.kaggle.yaml")
cmd = [sys.executable, "-u", "-m", "src.courtroom_rl.runner", "train",
       "--config", "configs/courtroom_rl.v4.kaggle.yaml", "--epochs", "1",
       "--batch-size", "8", "--dev-limit", "66", "--dev-every", "10",
       "--warmup-episodes", "8", "--seed", "0", "--log-steps"]
if RESUME:
    cmd += ["--resume", RESUME]
try:
    run(*cmd)
finally:
    output = Path("/kaggle/working/courtroom-training-output.tar.gz")
    with tarfile.open(output, "w:gz") as tar:
        for relative in ["runs", "cache/courtroom_rl/assessor.joblib"]:
            path = PROJECT / relative
            if path.exists():
                tar.add(path, arcname="courtroom-github/" + relative)
    print("Download:", output)
